# Week 1 In-Class Exercise: Does Healthcare Spending Predict Life Expectancy?

In the textbook (Chapter 1), we explored whether **GDP per capita predicts life satisfaction**.

In this exercise, you'll explore a similar question with a different dataset:

> **Does healthcare spending per capita predict life expectancy across countries?**

You'll follow the same workflow:
1. Load and explore the data
2. Visualize the relationship
3. Fit a linear regression model
4. Try a k-Nearest Neighbors model
5. Reflect on what you observe

**Data source:** [World Bank](https://data.worldbank.org/) — Health expenditure per capita (current US$) and Life expectancy at birth.

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/pjmcswee/IST707-Notebooks/blob/main/week1/week1_inclass_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor

plt.rc('font', size=12)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=12)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

## Step 1: Load the Data

We'll fetch the data directly from the World Bank API. The two indicators are:
- **SH.XPD.CHEX.PC.CD** — Current health expenditure per capita (US$)
- **SP.DYN.LE00.IN** — Life expectancy at birth (years)

In [ ]:
import urllib.request
import json

def fetch_world_bank_indicator(indicator, date_range="2019:2022"):
    """Fetch most recent value per country from World Bank API."""
    url = f"https://api.worldbank.org/v2/country/all/indicator/{indicator}?date={date_range}&format=json&per_page=2000"
    with urllib.request.urlopen(url) as resp:
        data = json.loads(resp.read())
    
    # Get list of actual countries (not regional aggregates)
    url_c = "https://api.worldbank.org/v2/country?per_page=400&format=json"
    with urllib.request.urlopen(url_c) as resp:
        countries = json.loads(resp.read())
    country_codes = {c['id'] for c in countries[1] if c['region']['id'] != 'NA'}
    
    # Most recent non-null value per country
    result = {}
    for record in data[1]:
        code = record['countryiso3code']
        if (record['value'] is not None and 
            code in country_codes and 
            code not in result):
            result[code] = {
                'country': record['country']['value'],
                'value': record['value']
            }
    return result

print("Fetching healthcare spending data...")
health_data = fetch_world_bank_indicator('SH.XPD.CHEX.PC.CD')
print(f"  {len(health_data)} countries")

print("Fetching life expectancy data...")
life_data = fetch_world_bank_indicator('SP.DYN.LE00.IN')
print(f"  {len(life_data)} countries")

## Step 2: Merge and Explore

In [ ]:
# Merge the two datasets
rows = []
for code in health_data:
    if code in life_data and health_data[code]['value'] > 0:
        rows.append({
            'Country': health_data[code]['country'],
            'Health Spending ($/capita)': round(health_data[code]['value'], 1),
            'Life Expectancy (years)': round(life_data[code]['value'], 2)
        })

df = pd.DataFrame(rows)
print(f"Combined dataset: {len(df)} countries")
df.describe()

In [ ]:
df.head(10)

## Step 3: Visualize the Relationship

**Task:** Create a scatter plot with healthcare spending on the x-axis and life expectancy on the y-axis.

What do you notice about the shape of the relationship?

In [ ]:
df.plot(kind='scatter', x='Health Spending ($/capita)', y='Life Expectancy (years)',
        figsize=(10, 6), alpha=0.6)
plt.title('Healthcare Spending vs. Life Expectancy')
plt.show()

### Discussion

The relationship isn't perfectly linear — it shows **diminishing returns**. 
Countries that spend very little see big life expectancy gains from small spending increases, 
but wealthy countries get less "bang for their buck."

This is a classic **logarithmic** relationship. Let's try plotting with log-scaled spending:

In [ ]:
df['Log Health Spending'] = np.log(df['Health Spending ($/capita)'])

df.plot(kind='scatter', x='Log Health Spending', y='Life Expectancy (years)',
        figsize=(10, 6), alpha=0.6)
plt.xlabel('Log(Health Spending per Capita)')
plt.title('Log Healthcare Spending vs. Life Expectancy')
plt.show()

Much more linear! This is a key insight in ML: sometimes **transforming your features** 
reveals a simpler relationship that models can capture more easily.

## Step 4: Fit a Linear Regression Model

**Task:** Fit a linear regression using log(spending) to predict life expectancy.

In [ ]:
X = df[['Log Health Spending']].values
y = df['Life Expectancy (years)'].values

model_lr = LinearRegression()
model_lr.fit(X, y)

print(f"Linear Regression")
print(f"  Intercept: {model_lr.intercept_:.2f}")
print(f"  Slope: {model_lr.coef_[0]:.2f}")
print(f"  R² score: {model_lr.score(X, y):.3f}")
print(f"\nInterpretation: Each doubling of healthcare spending is associated")
print(f"with roughly {model_lr.coef_[0] * np.log(2):.1f} additional years of life expectancy.")

In [ ]:
# Plot the regression line
plt.figure(figsize=(10, 6))
plt.scatter(df['Log Health Spending'], y, alpha=0.6, label='Countries')

X_range = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
plt.plot(X_range, model_lr.predict(X_range), 'r-', linewidth=2, label='Linear Regression')

plt.xlabel('Log(Health Spending per Capita)')
plt.ylabel('Life Expectancy (years)')
plt.title('Linear Regression: Log(Spending) → Life Expectancy')
plt.legend()
plt.show()

## Step 5: Try k-Nearest Neighbors

**Task:** Now fit a k-Nearest Neighbors regressor. How does it compare?

In [ ]:
model_knn = KNeighborsRegressor(n_neighbors=5)
model_knn.fit(X, y)

print(f"k-Nearest Neighbors (k=5)")
print(f"  R² score: {model_knn.score(X, y):.3f}")

In [ ]:
# Plot both models
plt.figure(figsize=(10, 6))
plt.scatter(df['Log Health Spending'], y, alpha=0.6, label='Countries')

X_range = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
plt.plot(X_range, model_lr.predict(X_range), 'r-', linewidth=2, label='Linear Regression')
plt.plot(X_range, model_knn.predict(X_range), 'g--', linewidth=2, label='k-NN (k=5)')

plt.xlabel('Log(Health Spending per Capita)')
plt.ylabel('Life Expectancy (years)')
plt.title('Comparing Models: Linear Regression vs k-NN')
plt.legend()
plt.show()

## Step 6: Make a Prediction

**Task:** A country spends $2,000 per capita on healthcare. What life expectancy does each model predict?

In [ ]:
new_spending = 2000
X_new = np.array([[np.log(new_spending)]])

pred_lr = model_lr.predict(X_new)[0]
pred_knn = model_knn.predict(X_new)[0]

print(f"For a country spending ${new_spending}/capita on healthcare:")
print(f"  Linear Regression predicts: {pred_lr:.1f} years")
print(f"  k-NN (k=5) predicts:        {pred_knn:.1f} years")

## Reflection Questions

Discuss with your neighbor or write brief answers:

1. **Why does the log transformation help?** What real-world phenomenon does it capture?

2. **Which model has a higher R² on this data — and is that necessarily better?** 
   Think about overfitting vs. underfitting.

3. **The US spends ~$12,000/capita but has a life expectancy of ~77 years** 
   (lower than many countries spending much less). What might explain this? 
   What does it suggest about the limits of this simple model?

4. **What other features** might you add to improve predictions?